# Practical Application III: Comparing Classifiers

**Overview**: In this practical application, your goal is to compare the performance of the classifiers we encountered in this section, namely K Nearest Neighbor, Logistic Regression, Decision Trees, and Support Vector Machines.  We will utilize a dataset related to marketing bank products over the telephone.  



### Getting Started

Our dataset comes from the UCI Machine Learning repository [link](https://archive.ics.uci.edu/ml/datasets/bank+marketing).  The data is from a Portugese banking institution and is a collection of the results of multiple marketing campaigns.  We will make use of the article accompanying the dataset [here](CRISP-DM-BANK.pdf) for more information on the data and features.



### Problem 1: Understanding the Data

To gain a better understanding of the data, please read the information provided in the UCI link above, and examine the **Materials and Methods** section of the paper.  How many marketing campaigns does this data represent?

<span style="color:cadetblue;"><b></b> 
The dataset used in this study was collected from a Portuguese bank that conducted direct marketing campaigns through its own contact center, mainly using telephone calls made by human agents, with occasional support from Internet banking. The data represents **17 marketing campaigns conducted between May 2008 and November 2010**, involving a total of **41188 customer contacts**. During these campaigns, customers were offered an attractive long-term deposit with good interest rates, and information about each contact was recorded, including customer characteristics, contact details, and whether the customer subscribed to the deposit. The main purpose of collecting and analyzing this data was to identify the characteristics associated with successful contacts and improve the efficiency of future marketing campaigns by reducing unnecessary contacts while maintaining a similar number of successful subscriptions.</span>

### Problem 2: Read in the Data

Use pandas to read in the dataset `bank-additional-full.csv` and assign to a meaningful variable name.

In [ ]:
import pandas as pd
import numpy as np
import time

import plotly.express as px
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score)

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

In [ ]:
bank_data = pd.read_csv("/Users/hudaalsaud/Desktop/ْUni/Uni/CV and workshops/Worshop_Training/Profesional training/Professional in machien learning/Project 3/module17_starter/data/bank-additional-full.csv", sep=";")

In [ ]:
bank_data.head()

### Problem 3: Understanding the Features


Examine the data description below, and determine if any of the features are missing values or need to be coerced to a different data type.


```
Input variables:
# bank client data:
1 - age (numeric)
2 - job : type of job (categorical: 'admin.','blue-collar','entrepreneur','housemaid','management','retired','self-employed','services','student','technician','unemployed','unknown')
3 - marital : marital status (categorical: 'divorced','married','single','unknown'; note: 'divorced' means divorced or widowed)
4 - education (categorical: 'basic.4y','basic.6y','basic.9y','high.school','illiterate','professional.course','university.degree','unknown')
5 - default: has credit in default? (categorical: 'no','yes','unknown')
6 - housing: has housing loan? (categorical: 'no','yes','unknown')
7 - loan: has personal loan? (categorical: 'no','yes','unknown')
# related with the last contact of the current campaign:
8 - contact: contact communication type (categorical: 'cellular','telephone')
9 - month: last contact month of year (categorical: 'jan', 'feb', 'mar', ..., 'nov', 'dec')
10 - day_of_week: last contact day of the week (categorical: 'mon','tue','wed','thu','fri')
11 - duration: last contact duration, in seconds (numeric). Important note: this attribute highly affects the output target (e.g., if duration=0 then y='no'). Yet, the duration is not known before a call is performed. Also, after the end of the call y is obviously known. Thus, this input should only be included for benchmark purposes and should be discarded if the intention is to have a realistic predictive model.
# other attributes:
12 - campaign: number of contacts performed during this campaign and for this client (numeric, includes last contact)
13 - pdays: number of days that passed by after the client was last contacted from a previous campaign (numeric; 999 means client was not previously contacted)
14 - previous: number of contacts performed before this campaign and for this client (numeric)
15 - poutcome: outcome of the previous marketing campaign (categorical: 'failure','nonexistent','success')
# social and economic context attributes
16 - emp.var.rate: employment variation rate - quarterly indicator (numeric)
17 - cons.price.idx: consumer price index - monthly indicator (numeric)
18 - cons.conf.idx: consumer confidence index - monthly indicator (numeric)
19 - euribor3m: euribor 3 month rate - daily indicator (numeric)
20 - nr.employed: number of employees - quarterly indicator (numeric)

Output variable (desired target):
21 - y - has the client subscribed a term deposit? (binary: 'yes','no')
```



In [ ]:
# Basic information about the dataset
print("Dataset shape:", bank_data.shape)

# Column names and data types
bank_data.info()

# Basic statistics for numerical variables
display(bank_data.describe())

# Number of unique values in each column
display(bank_data.nunique())

# Missing values
display(bank_data.isnull().sum())

# Count "unknown" values in each column
display((bank_data == "unknown").sum())

<span style="color:cadetblue;"><b>Data and Feature Overview:</b> 
Our initial review shows that the dataset contains **41,188 customer records and 21 features** describing different aspects of the customers and their interactions with the bank. These features include personal information, such as age and marital status, banking information, such as loans and credit status, details about previous and current marketing contacts, and broader economic indicators. The data is complete, with **no blank or technically missing values**, and the information is already stored in an appropriate format for analysis. The outcome we aim to understand is the **target variable y**, which records whether the customer **subscribed to the bank’s term deposit (“yes” or “no”)**. This provides a clear basis for examining which customer and campaign characteristics are associated with successful subscriptions.</span>


In [ ]:
# Categorical feature distribution

categorical_cols = bank_data.select_dtypes(include="object").columns

categorical_summary = []

for col in categorical_cols:
    counts = bank_data[col].value_counts(dropna=False)
    percentages = (counts / len(bank_data) * 100).round(2)

    for category, count in counts.items():
        categorical_summary.append({
            "Feature": col,
            "Category": category,
            "Count": count,
            "Percentage": percentages[category]
        })

categorical_summary = pd.DataFrame(categorical_summary)

display(categorical_summary)

<span style="color:cadetblue;"><b>Findings from Categorical Features:</b>
The categorical features contain a range of customer and campaign characteristics with different levels of representation. Some categories have very low representation, such as **default = yes** (0.01%) and **illiterate** education (0.04%), while others are highly concentrated, such as **poutcome = nonexistent** (86.34%). The target variable is imbalanced, with **88.73% of customers not subscribing** and **11.27% subscribing**. These observations will be considered in the following stages of the analysis.</span>


In [ ]:
# Numerical feature summary

numerical_cols = bank_data.select_dtypes(include=["int64", "float64"]).columns

numerical_summary = bank_data[numerical_cols].describe().T
numerical_summary["Unique"] = bank_data[numerical_cols].nunique()

numerical_summary = numerical_summary[
    ["count", "Unique", "mean", "std", "min", "25%", "50%", "75%", "max"]
].round(2)

display(numerical_summary)

<span style="color:cadetblue;"><b>Findings from Numerical Features:</b>
The numerical features show different patterns across customer and campaign information. **Age** ranges from 17 to 98 years, while **call duration** varies widely from 0 to 4,918 seconds. Most customers had no previous contacts, while a smaller group had previous contact history, with some customers contacted several times, up to 7. The economic measures show more limited variation. These differences will be examined further before making any data preparation decisions.</span>

In [ ]:
# Data Quality & Outliers

# Missing values
missing = bank_data.isnull().sum()

# Potential outliers using the IQR method
outlier_summary = []

for col in numerical_cols:
    Q1 = bank_data[col].quantile(0.25)
    Q3 = bank_data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((bank_data[col] < lower) | (bank_data[col] > upper)).sum()

    outlier_summary.append({
        "Feature": col,
        "Lower Bound": round(lower, 2),
        "Upper Bound": round(upper, 2),
        "Potential Outliers": outliers,
        "Percentage": round(outliers / len(bank_data) * 100, 2)
    })

outlier_summary = pd.DataFrame(outlier_summary)

display(outlier_summary)

<span style="color:cadetblue;"><b>Findings from Potential Outliers:</b>
The analysis identified some potential outliers in the numerical features. However, the unusual values in **campaign, pdays, and previous** can be explained by differences in customers’ contact history, including customers being contacted for the first time and others having several previous contacts. The age variable also contains a small number of potential outliers, including older customers, but these values can still represent valid customers. Overall, **no clear data errors or problematic outliers were identified that require correction at this stage**.</span>


In [ ]:
# Compare the subscription rate (Yes/No) across each categorical feature to identify patterns in customer behavior.

categorical_features = [
    "job", "marital", "education", "default",
    "housing", "loan", "contact", "month",
    "day_of_week", "poutcome"
]

for feature in categorical_features:
    rate = pd.crosstab(
        bank_data[feature],
        bank_data["y"],
        normalize="index"
    ) * 100

    rate = rate.round(2)
    rate.columns = ["No (%)", "Yes (%)"]

    print(f"\n--- {feature} ---")
    display(rate)

In [ ]:
# Visualize subscription rates across each categorical feature to compare customer groups.

categorical_features = [
    "job", "marital", "education", "default",
    "housing", "loan", "contact", "month",
    "day_of_week", "poutcome"
]

for feature in categorical_features:
    rate = pd.crosstab(
        bank_data[feature],
        bank_data["y"],
        normalize="index"
    ).reset_index()

    fig = px.bar(
        rate,
        x=feature,
        y=["no", "yes"],
        title=f"Subscription Rate by {feature}",
        barmode="stack"
    )

    fig.update_layout(
        yaxis_title="Proportion",
        yaxis_tickformat=".0%"
    )

    fig.show()

<span style="color:cadetblue;"><b>Categorical Features and Subscription:</b>
The results show clear differences in subscription rates for some features. **Job** varies noticeably, with higher rates among students (31.43%) and retired customers (25.23%). **Contact type** also differs, with cellular contacts showing a higher subscription rate (14.74%) than telephone contacts (5.23%).
The strongest differences appear in **month** and **previous campaign outcome**. Subscription rates are much higher in some months, while customers with a **successful previous campaign** have a substantially higher subscription rate (65.11%).
In contrast, **housing, loan, and day of week** show very similar rates across categories. </span>


In [ ]:
# Compare numerical customer characteristics between subscribers and non-subscribers.
numerical_features = [
    "age", "campaign", "pdays", "previous",
    "emp.var.rate", "cons.price.idx",
    "cons.conf.idx", "euribor3m", "nr.employed"
]

summary = bank_data.groupby("y")[numerical_features].agg(
    ["mean", "median", "min", "max"]
)

display(summary.round(2))

In [ ]:
# Visualize the distribution of numerical features across subscription outcomes.

numerical_features = [
    "age", "campaign", "pdays", "previous",
    "emp.var.rate", "cons.price.idx",
    "cons.conf.idx", "euribor3m", "nr.employed"
]

for feature in numerical_features:
    fig = px.box(
        bank_data,
        x="y",
        y=feature,
        points=False,
        title=f"{feature} by Subscription Outcome"
    )
    fig.show()

<span style="color:cadetblue;"><b>Numerical Features and Subscription:</b>
The numerical features show different relationships with subscription. **Age shows little difference** between customers who subscribed and those who did not, while **campaign, pdays, previous, and some economic indicators show more noticeable differences**. **pdays** requires special attention because **999** means the customer was not previously contacted, rather than representing an actual number of days. **Duration is excluded** because it is only known after the call and therefore is not suitable for a realistic prediction.</span>


In [ ]:
# Visualize the distribution of customers who subscribed versus those who did not.
fig = px.histogram(
    bank_data,
    x="y",
    text_auto=True,
    title="Distribution of Subscription Outcome"
)

fig.show()

In [ ]:
# Correlation Matrix — Numerical Features

corr = bank_data[numerical_features].corr().round(2)

display(corr)

In [ ]:
# Correlation Between Numerical Features

corr = bank_data[numerical_features].corr()

fig = px.imshow(
    corr,
    text_auto=".2f",
    aspect="auto",
    title="Correlation Between Numerical Features"
)

fig.show()

<span style="color:cadetblue;"><b>Findings:</b>
The correlation analysis shows strong relationships among the **economic features**, particularly between **emp.var.rate**, **euribor3m**, and **nr.employed**. This suggests that these variables provide overlapping information. Most other numerical features have weak relationships with each other, with a moderate relationship between **pdays** and **previous**.</span>


### Problem 4: Understanding the Task

After examining the description and data, your goal now is to clearly state the *Business Objective* of the task.  State the objective below.

<span style="color:cadetblue;"><b> Business Objective:</b>
The business objective is to improve the efficiency of the bank’s direct marketing campaigns by identifying customers who are more likely to subscribe to a term deposit. Using information collected from previous marketing contacts and customer characteristics, the aim is to develop predictive models that can distinguish between customers who are likely to subscribe and those who are not. Comparing different classification models allows us to determine which model provides the most useful predictions for supporting the bank’s targeting decisions. The expected business benefit is to focus telephone calls and other resources on a higher-quality group of potential customers, thereby reducing unnecessary contacts while maintaining or improving the number of successful subscriptions.</span>

### Problem 5: Engineering Features

Now that you understand your business objective, we will build a basic model to get started.  Before we can do this, we must work to encode the data.  Using just the bank information features, prepare the features and target column for modeling with appropriate encoding and transformations.

<span style="color:cadetblue;"><b> Feature Engineering Plan:</b>
In this step, we will prepare the data for modeling based on our previous findings. We will remove **duration** because it is not suitable for predicting the target, and remove **housing**, **loan**, and **day_of_week** because they showed little difference in subscription rates. We will then separate and encode the features and target (**no = 0, yes = 1**). After encoding, we will use the target relationship to decide which of the highly correlated economic features (**emp.var.rate, euribor3m, and nr.employed**) is most useful to keep. Finally, we will check the feature–target correlations, scale the numerical features where needed, and prepare the final dataset for modeling.</span>


In [ ]:
# Step 1: Remove unnecessary features
# duration is removed based on the project instructions.
# housing, loan, and day_of_week showed little difference in subscription rates.

bank_data = bank_data.drop(
    columns=["duration", "housing", "loan", "day_of_week"]
)

print("Remaining features:")
print(bank_data.columns.tolist())

In [ ]:
# Step 2: Encode categorical features and target

# Encode the target: no = 0, yes = 1
label_encoder = LabelEncoder()
bank_data["y"] = label_encoder.fit_transform(bank_data["y"])

# Show the target encoding
print("Target encoding:")
print(label_encoder.classes_)
print("no = 0, yes = 1")

# Identify categorical features
categorical_features = bank_data.select_dtypes(include="object").columns

# One-hot encode categorical features
bank_data = pd.get_dummies(bank_data,columns=categorical_features,drop_first=False)

# Convert True/False dummy values to 0/1
dummy_columns = bank_data.select_dtypes(include="bool").columns
bank_data[dummy_columns] = bank_data[dummy_columns].astype(int)

# Display the result
display(bank_data.head())

<span style="color:cadetblue;"><b></b>
In the next step, we compare the three highly correlated economic features **(emp.var.rate, euribor3m, and nr.employed)** with the target y. Since these features contain similar information, we use their correlation with the target to identify which one has the strongest relationship with subscription. This helps us decide which feature to keep and which redundant features can be removed.</span>

In [ ]:
# Step 3: Correlation matrix between economic features and the target

economic_features = [
    "emp.var.rate",
    "cons.price.idx",
    "cons.conf.idx",
    "euribor3m",
    "nr.employed",
    "y"]

# Compute the correlation matrix
economic_corr = bank_data[economic_features].corr()

# Display the correlation matrix
print("Correlation Matrix: Economic Features and Target (y)")
display(economic_corr)

# Display only the correlation with the target
#target_corr = economic_corr["y"].drop("y").sort_values(key=abs,ascending=False)

#print("\nCorrelation of Economic Features with Target (y):")
#display(target_corr)

In [ ]:
# Plot the correlation matrix

fig = px.imshow(
    economic_corr,
    text_auto=".2f",
    aspect="auto",
    title="Correlation Between Economic Features and Target"
)

fig.show()

# Show the correlation of each economic feature with the target
target_corr = economic_corr["y"].drop("y").sort_values(
    key=abs,
    ascending=False
)

print("\nCorrelation of Economic Features with Target (y):")
display(target_corr.to_frame("Correlation with y"))

<span style="color:cadetblue;"><b>Decision: Remove Redundant Economic Features</b>
The correlation analysis shows that emp.var.rate, euribor3m, and nr.employed contain very similar information, with correlations as high as 0.97. Among them, nr.employed has the strongest relationship with the target y (-0.35). Therefore, we will keep nr.employed and remove emp.var.rate and euribor3m to reduce redundancy while retaining the most useful feature.</span>

In [ ]:
# Remove redundant economic features

bank_data = bank_data.drop(columns=["emp.var.rate", "euribor3m"])

# Check the remaining features
print("Remaining features:")
print(bank_data.columns.tolist())

In [ ]:
# Step 4: Correlation Between Encoded Features and Target

# Compute the correlation matrix
corr_encoded = bank_data.corr()

# Display the correlation of all features with y
target_corr = corr_encoded["y"].sort_values(
    key=abs,
    ascending=False
)

print("Correlation Between All Features and Target (y):")
display(target_corr.to_frame("Correlation with y"))



### Problem 6: Train/Test Split

With your data prepared, split it into a train and test set.

In [ ]:
# Separate Features and Target, Then Split the Data

# Separate features (X) and target (y)
X = bank_data.drop("y", axis=1)
y = bank_data["y"]

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Check the sizes
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

In [ ]:
# Scale Numerical Features for KNN, Logistic Regression, and SVM

# Numerical features to scale
numerical_features = [
    "age",
    "campaign",
    "pdays",
    "previous",
    "cons.price.idx",
    "cons.conf.idx",
    "nr.employed"
]

# Create copies so the original X_train and X_test remain unchanged
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Create the scaler
scaler = StandardScaler()

# Fit the scaler only on the training data
X_train_scaled[numerical_features] = scaler.fit_transform(
    X_train[numerical_features]
)

# Use the same scaler to transform the test data
X_test_scaled[numerical_features] = scaler.transform(
    X_test[numerical_features]
)

print("Scaled training data:", X_train_scaled.shape)
print("Scaled testing data:", X_test_scaled.shape)

### Problem 7: A Baseline Model

Before we build our first model, we want to establish a baseline.  What is the baseline performance that our classifier should aim to beat?

In [ ]:
# Create the baseline model
dummy_model = DummyClassifier()

# Measure training time
start_time = time.time()

dummy_model.fit(X_train, y_train)

train_time = time.time() - start_time

# Predictions
y_train_pred = dummy_model.predict(X_train)
y_test_pred = dummy_model.predict(X_test)

# Calculate metrics
baseline_train_accuracy = accuracy_score(y_train, y_train_pred)
baseline_test_accuracy = accuracy_score(y_test, y_test_pred)

baseline_precision = precision_score(y_test, y_test_pred, zero_division=0)
baseline_recall = recall_score(y_test, y_test_pred, zero_division=0)
baseline_f1 = f1_score(y_test, y_test_pred, zero_division=0)


# Display results
print("Baseline Model: Dummy Classifier")
print(f"Train Time: {train_time:.4f} seconds")
print(f"Train Accuracy: {baseline_train_accuracy:.4f}")
print(f"Test Accuracy:  {baseline_test_accuracy:.4f}")
print(f"Precision:  {baseline_precision:.4f}")
print(f"Recall:     {baseline_recall:.4f}")
print(f"F1 Score:   {baseline_f1:.4f}")

<span style="color:cadetblue;"><b></b>
The baseline achieved **88.74%** accuracy, but precision, recall, and F1-score were **0%** because it did not identify any subscribers. It therefore provides a reference point for comparing the other models.
</span>

### Problem 8: A Simple Model

Use Logistic Regression to build a basic model on your data.  

In [ ]:
# Build basic Logistic Regression 
lr_model_default = LogisticRegression()

# Fit the model and measure training time
start_time = time.time()
lr_model_default.fit(X_train_scaled, y_train)
lr_train_time = time.time() - start_time


In [ ]:
# Examine the Logistic Regression coefficients

# Create a DataFrame showing each feature and its coefficient
coefficients = pd.DataFrame({
    "Feature": X_train_scaled.columns,
    "Coefficient": lr_model_default.coef_[0]
})

# Sort by coefficient magnitude
coefficients["Absolute Coefficient"] = coefficients["Coefficient"].abs()
coefficients = coefficients.sort_values(
    "Absolute Coefficient",
    ascending=False
)

print("Intercept:", lr_model_default.intercept_[0])
print(coefficients)

In [ ]:
# Plot Logistic Regression coefficients

plt.figure(figsize=(10, 12))

plt.barh(coefficients["Feature"], coefficients["Coefficient"])

plt.xlabel("Coefficient")
plt.ylabel("Feature")
plt.title("Logistic Regression Coefficients")
plt.axvline(0, linewidth=1)

plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Obtain predicted probabilities for the test data

y_test_prob = lr_model_default.predict_proba(X_test_scaled)[:, 1]

print("First 10 predicted probabilities:")
print(y_test_prob[:10])

<span style="color:cadetblue;"><b></b>
The predicted probability shows how likely each customer is to say “Yes.” For example, 0.06 means an estimated 6% chance, while 0.49 means 49%. This helps the bank prioritize customers who are more likely to respond positively, making the campaign more efficient by focusing time and resources on the most promising customers.</span>

In [ ]:
# Convert 0/1 predictions into Yes/No labels

y_test_pred_labels = np.where(y_test_pred == 1, "Yes", "No")

print("First 10 predicted probabilities:")
print(y_test_prob[:10])

print("\nFirst 10 predicted classes:")
print(y_test_pred_labels[:10])


### Problem 9: Score the Model

What is the accuracy of your model?

In [ ]:
# Predict the training and test sets
y_train_pred = lr_model_default.predict(X_train_scaled)
y_test_pred = lr_model_default.predict(X_test_scaled)


# Calculate performance metrics
lr_train_accuracy = accuracy_score(y_train, y_train_pred)
lr_test_accuracy = accuracy_score(y_test, y_test_pred)

lr_precision = precision_score(y_test, y_test_pred)
lr_recall = recall_score(y_test, y_test_pred)
lr_f1 = f1_score(y_test, y_test_pred)


print(f"Train Accuracy: {lr_train_accuracy:.4f}")
print(f"Test Accuracy:  {lr_test_accuracy:.4f}")
print(f"Precision:      {lr_precision:.4f}")
print(f"Recall:         {lr_recall:.4f}")
print(f"F1-score:       {lr_f1:.4f}")

<span style="color:cadetblue;"><b></b>
The Logistic Regression model correctly classified **90.17% of customers**, which is slightly better than the baseline accuracy of **88.74%**. However, because the dataset is imbalanced, accuracy alone does not give a complete picture of the model's performance. When the model predicts that a customer will subscribe, it is correct about **70.34% of the time**, which indicates relatively good precision. On the other hand, the model identifies only **21.98% of the customers who actually subscribe**, meaning that many potential subscribers are missed. The **F1-score of 33.50%** reflects the difficulty of achieving a good balance between identifying subscribers and avoiding incorrect predictions. Overall, the model shows reasonable accuracy and precision, but its low recall suggests that there is room for improvement.</span>

### Problem 10: Model Comparisons

Now, we aim to compare the performance of the Logistic Regression model to our KNN algorithm, Decision Tree, and SVM models.  Using the default settings for each of the models, fit and score each.  Also, be sure to compare the fit time of each of the models.  Present your findings in a `DataFrame` similar to that below:

| Model | Train Time | Train Accuracy | Test Accuracy |
| ----- | ---------- | -------------  | -----------   |
|     |    |.     |.     |

In [ ]:
# Build the KNN model using default settings
knn_model = KNeighborsClassifier()

# Fit the model and measure training time
start_time = time.time()
knn_model.fit(X_train_scaled, y_train)
knn_train_time = time.time() - start_time

# Predict training and test sets
knn_train_pred = knn_model.predict(X_train_scaled)
knn_test_pred = knn_model.predict(X_test_scaled)

# Calculate accuracy
knn_train_accuracy = accuracy_score(y_train, knn_train_pred)
knn_test_accuracy = accuracy_score(y_test, knn_test_pred)

# Calculate precision and recall and F1
knn_precision = precision_score(y_test, knn_test_pred)
knn_recall = recall_score(y_test, knn_test_pred)
knn_f1 = f1_score(y_test, knn_test_pred)

In [ ]:
# Build the Decision Tree using default settings
dt_model = DecisionTreeClassifier()

# Fit the model and measure training time
start_time = time.time()
dt_model.fit(X_train, y_train)
dt_train_time = time.time() - start_time

# Predict training and test sets
dt_train_pred = dt_model.predict(X_train)
dt_test_pred = dt_model.predict(X_test)

# Calculate accuracy
dt_train_accuracy = accuracy_score(y_train, dt_train_pred)
dt_test_accuracy = accuracy_score(y_test, dt_test_pred)

# Calculate precision, recall, and F1-score
dt_precision = precision_score(y_test, dt_test_pred)
dt_recall = recall_score(y_test, dt_test_pred)
dt_f1 = f1_score(y_test, dt_test_pred)

In [ ]:
# Build the SVM using default settings
svm_model = SVC()

# Fit the model and measure training time
start_time = time.time()
svm_model.fit(X_train_scaled, y_train)
svm_train_time = time.time() - start_time

# Predict training and test sets
svm_train_pred = svm_model.predict(X_train_scaled)
svm_test_pred = svm_model.predict(X_test_scaled)

# Calculate accuracy
svm_train_accuracy = accuracy_score(y_train, svm_train_pred)
svm_test_accuracy = accuracy_score(y_test, svm_test_pred)

# Calculate precision, recall, and F1-score
svm_precision = precision_score(y_test, svm_test_pred)
svm_recall = recall_score(y_test, svm_test_pred)
svm_f1 = f1_score(y_test, svm_test_pred)


In [ ]:
# Building DataFrame for model performance comparison

comparison_df = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Train Time": lr_train_time,
        "Train Accuracy": lr_train_accuracy,
        "Test Accuracy": lr_test_accuracy,
        "Precision": lr_precision,
        "Recall": lr_recall,
        "F1-score": lr_f1
    },
    {
        "Model": "KNN",
        "Train Time": knn_train_time,
        "Train Accuracy": knn_train_accuracy,
        "Test Accuracy": knn_test_accuracy,
        "Precision": knn_precision,
        "Recall": knn_recall,
        "F1-score": knn_f1
    },
    {
        "Model": "Decision Tree",
        "Train Time": dt_train_time,
        "Train Accuracy": dt_train_accuracy,
        "Test Accuracy": dt_test_accuracy,
        "Precision": dt_precision,
        "Recall": dt_recall,
        "F1-score": dt_f1
    },
    {
        "Model": "SVM",
        "Train Time": svm_train_time,
        "Train Accuracy": svm_train_accuracy,
        "Test Accuracy": svm_test_accuracy,
        "Precision": svm_precision,
        "Recall": svm_recall,
        "F1-score": svm_f1
    }
])

display(comparison_df.round(4))

In [ ]:
# plotting the test accuracy and F1-score of the four classification models.

fig = px.bar(
    comparison_df,
    x="Model",
    y=["Test Accuracy", "F1-score"],
    barmode="group",
    title="Test Accuracy and F1-score Comparison",
    labels={
        "value": "Score",
        "variable": "Metric",
        "Model": "Model"
    },
    text_auto=".3f"
)

fig.update_layout(
    yaxis=dict(range=[0, 1]),
    legend_title="Metric"
)

fig.show()

### Problem 11: Improving the Model

Now that we have some basic models on the board, we want to try to improve these.  Below, we list a few things to explore in this pursuit.


- Hyperparameter tuning and grid search.  All of our models have additional hyperparameters to tune and explore.  For example the number of neighbors in KNN or the maximum depth of a Decision Tree.  
- Adjust your performance metric

<span style="color:cadetblue;"><b></b>
In this part, we aim to improve the performance of our classification models by tuning their hyperparameters and selecting the settings that provide the best results. We use Grid Search with cross-validation to explore different hyperparameter values. Because our target variable is imbalanced, we use the F1-score as the main metric for selecting the best model, as it provides a balance between precision and recall. After tuning each model, we calculate and compare the training time, train accuracy, test accuracy, precision, recall, and F1-score to evaluate the overall performance of the improved models.</span>

In [ ]:
# Tune the Logistic Regression Model

# Create the Logistic Regression model
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Define the hyperparameters to search
param_grid = {
    "C": [ 0.01, 0.1, 1, 5, 10],
    "class_weight": [None, "balanced"]
}


# Measure training time
start_time = time.time()

# Perform Grid Search with 5-fold Cross-Validation
# F1 is used to select the best model because the data is imbalanced
grid = GridSearchCV(
    estimator=lr_model,
    param_grid=param_grid,
    cv=5,
    scoring="f1"

)

# Train the models
grid.fit(X_train_scaled, y_train)

training_time = time.time() - start_time

# Get the best model
best_lr_model = grid.best_estimator_

# Predict the training and test sets
y_train_pred = best_lr_model.predict(X_train_scaled)
y_test_pred = best_lr_model.predict(X_test_scaled)

# Calculate performance metrics
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

test_precision = precision_score(y_test, y_test_pred, zero_division=0)
test_recall = recall_score(y_test, y_test_pred, zero_division=0)
test_f1 = f1_score(y_test, y_test_pred, zero_division=0)

# Create results table
lr_results = pd.DataFrame([{
    "Model": "Logistic Regression",
    "Best Parameters": grid.best_params_,
    "CV F1 Score": grid.best_score_,
    "Train Time": training_time,
    "Train Accuracy": train_accuracy,
    "Test Accuracy": test_accuracy,
    "Precision": test_precision,
    "Recall": test_recall,
    "F1 Score": test_f1
}])

display(lr_results)

<span style="color:cadetblue;"><b></b>
To further improve the Logistic Regression model, we add L1 and L2 regularization to the Grid Search and compare their performance. L1 regularization can reduce some feature coefficients to zero, allowing us to identify which features are selected and which are not selected by the model. We then compare the selected features and the model performance to determine which regularization approach works best.</span>

In [ ]:
# Tune Logistic Regression with L1 Regularization

lr_model_l1 = LogisticRegression(
    solver="liblinear",
    penalty="l1",
    max_iter=1000,
    random_state=42
)


lr_param_grid = {
    "C": [0.01, 0.1, 1, 5, 10],
    "class_weight": [None, "balanced"]
}


start_time = time.time()

lr_grid_l1 = GridSearchCV(
    estimator=lr_model_l1,
    param_grid=lr_param_grid,
    cv=5,
    scoring="f1"
)

lr_grid_l1.fit(X_train_scaled, y_train)

lr_l1_training_time = time.time() - start_time

best_lr_l1_model = lr_grid_l1.best_estimator_

lr_l1_train_pred = best_lr_l1_model.predict(X_train_scaled)
lr_l1_test_pred = best_lr_l1_model.predict(X_test_scaled)

lr_l1_train_accuracy = accuracy_score(y_train, lr_l1_train_pred)

lr_l1_test_accuracy = accuracy_score(y_test, lr_l1_test_pred)

lr_l1_precision = precision_score(y_test, lr_l1_test_pred, zero_division=0)

lr_l1_recall = recall_score(y_test, lr_l1_test_pred, zero_division=0)

lr_l1_f1 = f1_score(y_test, lr_l1_test_pred, zero_division=0)

lr_l1_results = pd.DataFrame([{
    "Model": "Logistic Regression (L1)",
    "Best Parameters": lr_grid_l1.best_params_,
    "CV F1 Score": lr_grid_l1.best_score_,
    "Train Time": lr_l1_training_time,
    "Train Accuracy": lr_l1_train_accuracy,
    "Test Accuracy": lr_l1_test_accuracy,
    "Precision": lr_l1_precision,
    "Recall": lr_l1_recall,
    "F1 Score": lr_l1_f1
}])

print("LOGISTIC REGRESSION L1 RESULTS")
display(lr_l1_results)

# Feature selection
lr_l1_coefficients = pd.DataFrame({
    "Feature": X_train_scaled.columns,
    "Coefficient": best_lr_l1_model.coef_[0]
})

lr_l1_coefficients["Selected"] = (lr_l1_coefficients["Coefficient"] != 0)

selected_count = lr_l1_coefficients["Selected"].sum()
not_selected_count = ( ~lr_l1_coefficients["Selected"]).sum()

print("\nFEATURE SELECTION RESULTS")
print(f"Selected features:     {selected_count}")
print(f"Not selected features: {not_selected_count}")

lr_l1_coefficients = lr_l1_coefficients.sort_values( "Coefficient", ascending=False)

display(lr_l1_coefficients)

In [ ]:
# Tune the Decision Tree Model

# Create the Decision Tree model
dt_model = DecisionTreeClassifier(random_state=42)

# Define the hyperparameter to search

dt_param_grid = {
    "max_depth": [None, 3, 5, 10],
    "min_samples_leaf": [1, 2, 5, 10],
     "class_weight": [None, "balanced"]
}


# Measure training time
start_time = time.time()

# Perform Grid Search with 5-fold Cross-Validation
# F1 is used because the target is imbalanced
dt_grid = GridSearchCV(
    estimator=dt_model,
    param_grid=dt_param_grid,
    cv=5,
    scoring="f1"
)

# Train the models
dt_grid.fit(X_train, y_train)

dt_training_time = time.time() - start_time

# Get the best model
best_dt_model = dt_grid.best_estimator_

# Make predictions
dt_train_pred = best_dt_model.predict(X_train)
dt_test_pred = best_dt_model.predict(X_test)

# Calculate performance metrics
dt_train_accuracy = accuracy_score(y_train, dt_train_pred)
dt_test_accuracy = accuracy_score(y_test, dt_test_pred)
dt_precision = precision_score(y_test, dt_test_pred, zero_division=0)
dt_recall = recall_score(y_test, dt_test_pred, zero_division=0)
dt_f1 = f1_score(y_test, dt_test_pred, zero_division=0)

# Create results table
dt_results = pd.DataFrame([{
    "Model": "Decision Tree",
    "Best Parameters": dt_grid.best_params_,
    "CV F1 Score": dt_grid.best_score_,
    "Train Time": dt_training_time,
    "Train Accuracy": dt_train_accuracy,
    "Test Accuracy": dt_test_accuracy,
    "Precision": dt_precision,
    "Recall": dt_recall,
    "F1 Score": dt_f1
}])

display(dt_results)

In [ ]:
# Tune the KNN Model

# Create the KNN model
knn_model = KNeighborsClassifier()

# Define the hyperparameter to search
knn_param_grid = {
    "n_neighbors": [3, 5, 7, 11],
    "weights": ["uniform", "distance"]
}

# Measure training time
start_time = time.time()

# Perform Grid Search with 5-fold Cross-Validation
# F1 is used because the target is imbalanced
knn_grid = GridSearchCV(
    estimator=knn_model,
    param_grid=knn_param_grid,
    cv=5,
    scoring="f1"
)

# Train the models
knn_grid.fit(X_train_scaled, y_train)

knn_training_time = time.time() - start_time

# Get the best model
best_knn_model = knn_grid.best_estimator_

# Make predictions
knn_train_pred = best_knn_model.predict(X_train_scaled)
knn_test_pred = best_knn_model.predict(X_test_scaled)

# Calculate performance metrics
knn_train_accuracy = accuracy_score(y_train, knn_train_pred)
knn_test_accuracy = accuracy_score(y_test, knn_test_pred)

knn_precision = precision_score(y_test, knn_test_pred, zero_division=0)
knn_recall = recall_score(y_test, knn_test_pred, zero_division=0)
knn_f1 = f1_score(y_test, knn_test_pred, zero_division=0)

# Create results table
knn_results = pd.DataFrame([{
    "Model": "KNN",
    "Best Parameters": knn_grid.best_params_,
    "CV F1 Score": knn_grid.best_score_,
    "Train Time": knn_training_time,
    "Train Accuracy": knn_train_accuracy,
    "Test Accuracy": knn_test_accuracy,
    "Precision": knn_precision,
    "Recall": knn_recall,
    "F1 Score": knn_f1
}])

display(knn_results)

<span style="color:cadetblue;"><b></b>
For the next step, we use 25% of the training data for the SVM Grid Search because training an SVM on the full dataset takes a significant amount of time. The purpose of this step is only to find the best combination of hyperparameters, including the kernel and C and gamma values. Once the best parameters are identified, we will use them to train and refine the final SVM model on the full training dataset in the next cell.</span>

In [ ]:
# Tune the SVM Model with sample

# Take a stratified sample from the training data
X_train_svm, _, y_train_svm, _ = train_test_split(
    X_train_scaled,
    y_train,
    train_size=0.25,
    random_state=42,
    stratify=y_train
)

svm_model = SVC()

svm_param_grid = [
    {
        "kernel": ["linear"],
        "C": [0.1, 5, 7, 10],
        "class_weight": [None, "balanced"]
    },
    {
        "kernel": ["rbf"],
        "C": [0.1, 5, 7, 10],
        "gamma": [0.01, 0.1, 1],
        "class_weight": [None, "balanced"]
    }
]

svm_grid = GridSearchCV(
    estimator=svm_model,
    param_grid=svm_param_grid,
    cv=3,
    scoring="f1"
)

svm_grid.fit(X_train_svm, y_train_svm)

print("Best Parameters:", svm_grid.best_params_)
print("Best CV F1 Score:", svm_grid.best_score_)

In [ ]:
# Refine the SVM Model Using the Full Training Set

svm_model_final = SVC(C=7,gamma=0.1, kernel="rbf", class_weight= "balanced")

start_time = time.time()

svm_model_final.fit(X_train_scaled, y_train)

svm_training_time = time.time() - start_time

svm_train_pred = svm_model_final.predict(X_train_scaled)
svm_test_pred = svm_model_final.predict(X_test_scaled)

svm_train_accuracy = accuracy_score(y_train, svm_train_pred)
svm_test_accuracy = accuracy_score(y_test, svm_test_pred)
svm_precision = precision_score(y_test, svm_test_pred, zero_division=0)
svm_recall = recall_score(y_test, svm_test_pred, zero_division=0)
svm_f1 = f1_score( y_test, svm_test_pred, zero_division=0)

svm_results = pd.DataFrame([{
    "Model": "SVM",
    "Train Time": svm_training_time,
    "Train Accuracy": svm_train_accuracy,
    "Test Accuracy": svm_test_accuracy,
    "Precision": svm_precision,
    "Recall": svm_recall,
    "F1 Score": svm_f1
}])

print("SVM RESULTS")
display(svm_results)

In [ ]:
# Compare all classification models

all_models_results = pd.DataFrame([
    {
        "Model": "Logistic Regression (L2)",
        "Train Time": training_time,
        "Train Accuracy": train_accuracy,
        "Test Accuracy": test_accuracy,
        "Precision": test_precision,
        "Recall": test_recall,
        "F1 Score": test_f1
    },
    {
        "Model": "Logistic Regression (L1)",
        "Train Time": lr_l1_training_time,
        "Train Accuracy": lr_l1_train_accuracy,
        "Test Accuracy": lr_l1_test_accuracy,
        "Precision": lr_l1_precision,
        "Recall": lr_l1_recall,
        "F1 Score": lr_l1_f1
    },
    {
        "Model": "KNN",
        "Train Time": knn_training_time,
        "Train Accuracy": knn_train_accuracy,
        "Test Accuracy": knn_test_accuracy,
        "Precision": knn_precision,
        "Recall": knn_recall,
        "F1 Score": knn_f1
    },
    {
        "Model": "Decision Tree",
        "Train Time": dt_training_time,
        "Train Accuracy": dt_train_accuracy,
        "Test Accuracy": dt_test_accuracy,
        "Precision": dt_precision,
        "Recall": dt_recall,
        "F1 Score": dt_f1
    },
    {
        "Model": "SVM",
        "Train Time": svm_training_time,
        "Train Accuracy": svm_train_accuracy,
        "Test Accuracy": svm_test_accuracy,
        "Precision": svm_precision,
        "Recall": svm_recall,
        "F1 Score": svm_f1
    }
])

display(all_models_results)

In [ ]:
# Compare Test Accuracy and F1 Score for all classification models

fig = px.bar(
    all_models_results,
    x="Model",
    y=["Test Accuracy", "F1 Score"],
    barmode="group",
    title="Test Accuracy vs F1 Score",
    labels={
        "value": "Score",
        "variable": "Metric"
    },
    text_auto=".3f"
)

fig.update_layout(
    yaxis=dict(range=[0, 1]),
    xaxis_title="Model",
    yaxis_title="Score"
)

fig.show()

<span style="color:cadetblue;"><b></b>
Given the class imbalance in the dataset, the F1-score was prioritized over accuracy when selecting the final model. Although KNN achieved the highest test accuracy (89.99%) and precision (61.37%), its recall was only 29.96%, meaning that it identified only about 30% of the customers who actually subscribed to the term deposit. In contrast, the Decision Tree identified about 63% of the customers who actually subscribed to the term deposit (recall = 62.82%). Its precision was 39.05%, meaning that about 39% of the customers identified by the model as potential subscribers actually subscribed to the term deposit. This balance between identifying more actual subscribers and making relevant customer selections resulted in the highest F1-score among the evaluated models (0.4816). From a business perspective, the Decision Tree is therefore preferable because it gives the bank a better opportunity to identify customers who are likely to subscribe, reducing the number of potential subscribers that may be missed. Although KNN provides more precise recommendations, it identifies only about 30% of the customers who actually subscribe. Therefore, the Decision Tree was selected as the best-performing model based on the project's priority of identifying the minority positive class.</span>

In [ ]:
### Predict a New Customer Using the Decision Tree Model

def predict_new_customer_DT():

    print("=" * 60)
    print("             NEW CUSTOMER INFORMATION")
    print("=" * 60)

    customer = {
        "age": float(input("Age: ")),
        "job": input("Job: "),
        "marital": input("Marital status: "),
        "education": input("Education: "),
        "default": input("Credit in default (yes/no): "),
        "contact": input("Contact method: "),
        "month": input("Last contact month: "),
        "campaign": float(input("Number of contacts during this campaign: ")),
        "pdays": float(input("Days since previous contact: ")),
        "previous": float(input("Number of previous contacts: ")),
        "poutcome": input("Outcome of previous campaign: "),
        "cons.price.idx": float(input("Consumer price index: ")),
        "cons.conf.idx": float(input("Consumer confidence index: ")),
        "nr.employed": float(input("Number of employees indicator: "))
    }

    # Convert customer information into a DataFrame
    new_customer = pd.DataFrame([customer])

    # One-hot encode categorical features
    new_customer = pd.get_dummies(new_customer)

    # Match the training features
    new_customer = new_customer.reindex(
        columns=X_train_scaled.columns,
        fill_value=0
    )

    # Scale numerical features using the existing scaler
    new_customer[numerical_features] = scaler.transform(
        new_customer[numerical_features]
    )

    # Make the prediction
    prediction = best_dt_model.predict(new_customer)[0]

    # Get probabilities for both outcomes
    probabilities = best_dt_model.predict_proba(new_customer)[0]

    no_probability = probabilities[0]
    yes_probability = probabilities[1]

    prediction_label = "Yes" if prediction == 1 else "No"

    # Display results
    print("\n" + "=" * 60)
    print("                 PREDICTION RESULT")
    print("=" * 60)

    print("\nCUSTOMER INFORMATION")
    print("-" * 60)

    for feature, value in customer.items():
        print(f"{feature}: {value}")

    print("\nPREDICTION")
    print("-" * 60)

    print(f"Estimated probability of subscribing: {yes_probability:.1%}")
    print(f"Estimated probability of not subscribing: {no_probability:.1%}")
    print(f"Prediction for subscribing: {prediction_label}")

    print("\nINTERPRETATION")
    print("-" * 60)

    if prediction_label == "Yes":
        print(
            "The model predicts that this customer is likely "
            "to subscribe to the term deposit."
        )
    else:
        print(
            "The model predicts that this customer is unlikely "
            "to subscribe to the term deposit."
        )

    print("=" * 60)


# Run the prediction
predict_new_customer_DT()

##### Questions